# HW1: Frame-Level Speech Recognition

This homework works with MFCC data consisting of 28 features at each time step/frame. The final model is able to recognize the phoneme occured in that frame.

#### Project Title

This project involves experimenting with various neural network architectures to achieve optimal performance. Below are the running instructions, architectures tried, and best configurations.

##### Running Instructions
To run this project, simply execute the notebook:


##### Configuration Overview

Below is the updated configuration used for this project:

- **Number of Epochs:** 40
- **Batch Size:** 8192
- **Context Size:** 25
- **Initial Learning Rate:** 0.003 (tried 0.005,0.001,0.002)

###### Model Parameters:
- **Architecture:** MLP (Multi-Layer Perceptron)
- **Dropout Rate:** 0.2 (tried 0.5,0.4,0.3)
- **Time Mask:** 10
- **Frequency Mask:** 5

###### Scheduler Parameters:
- **Scheduler Type:** Cosine Annealing
  - **T_max:** 10 (Maximum number of epochs for cosine annealing)
  - **Minimum Learning Rate (eta_min):** 1e-5

###### Optimizer Parameters:
- **Optimizer Type:** Adam
- **Weight Decay:** 1e-4 (Regularization parameter)

###### Weight Initialization:
- **Method:** Kaiming Uniform
- **Nonlinearity:** GELU (Used for activation during initialization)

##### Best Combination

Based on experiments, the following combination yielded the best performance:

- **Architecture:** 8 layers of Diamond with GELU activation and batch normalization
- **Dropout Rate:** 0.2
- **Learning Rate Scheduler:** Cosine Annealing
- **Weight Initialization:** Kaiming
- **Context Size:** 25
- **Batch Size:** 8192


##### Network Configuration
Here is the best-performing network architecture:

Network(
  (model): Sequential(
    (0): Linear(in_features=1428, out_features=512, bias=True)

    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=512, out_features=1024, bias=True)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=1024, out_features=2048, bias=True)
    (8): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): GELU(approximate='none')
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=2048, out_features=3000, bias=True)
    (12): BatchNorm1d(3000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): GELU(approximate='none')
    (14): Dropout(p=0.2, inplace=False)
    (15): Linear(in_features=3000, out_features=2048, bias=True)
    (16): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (17): GELU(approximate='none')
    (18): Dropout(p=0.2, inplace=False)
    (19): Linear(in_features=2048, out_features=1024, bias=True)
    (20): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (21): GELU(approximate='none')
    (22): Dropout(p=0.2, inplace=False)
    (23): Linear(in_features=1024, out_features=512, bias=True)
    (24): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (25): GELU(approximate='none')
    (26): Dropout(p=0.2, inplace=False)
    (27): Linear(in_features=512, out_features=42, bias=True)
  )
)


# Libraries

In [2]:
!pip install torchsummaryX==1.1.0 wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.8 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
from torchsummaryX import summary
import sklearn
import gc
import zipfile
import pandas as pd
from tqdm.auto import tqdm
import os
import datetime
import wandb
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


In [4]:
''' If you are using colab, you can import google drive to save model checkpoints in a folder
    If you want to use it, uncomment the two lines below
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
### PHONEME LIST
PHONEMES = [
            '[SIL]',   'AA',    'AE',    'AH',    'AO',    'AW',    'AY',
            'B',     'CH',    'D',     'DH',    'EH',    'ER',    'EY',
            'F',     'G',     'HH',    'IH',    'IY',    'JH',    'K',
            'L',     'M',     'N',     'NG',    'OW',    'OY',    'P',
            'R',     'S',     'SH',    'T',     'TH',    'UH',    'UW',
            'V',     'W',     'Y',     'Z',     'ZH',    '[SOS]', '[EOS]']

# Kaggle

This section contains code that helps you install kaggle's API, creating kaggle.json with you username and API key details. Make sure to input those in the given code to ensure you can download data from the competition successfully.

In [6]:
!pip install --upgrade --force-reinstall --no-deps kaggle

  Using cached kaggle-1.6.17-py3-none-any.whl
  Attempting uninstall: kaggle
    Found existing installation: kaggle 1.6.17
    Uninstalling kaggle-1.6.17:
      Successfully uninstalled kaggle-1.6.17


In [7]:

!mkdir /root/.kaggle

with open("/root/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"","key":""}')
    # Put your kaggle username & key here

!chmod 600 /root/.kaggle/kaggle.json

In [8]:
# commands to download data from kaggle
!kaggle competitions download -c 11785-hw1p2-f24

!unzip -qo /content/11785-hw1p2-f24.zip -d '/content'

100% 3.98G/3.98G [00:22<00:00, 245MB/s]
100% 3.98G/3.98G [00:22<00:00, 190MB/s]


# Dataset

This section covers the dataset/dataloader class for speech data. You will have to spend time writing code to create this class successfully. We have given you a lot of comments guiding you on what code to write at each stage, from top to bottom of the class. Please try and take your time figuring this out, as it will immensely help in creating dataset/dataloader classes for future homeworks.

Before running the following cells, please take some time to analyse the structure of data. Try loading a single MFCC and its transcipt, print out the shapes and print out the values. Do the transcripts look like phonemes?

In [9]:
# Dataset class to load train and validation data

class AudioDataset(torch.utils.data.Dataset):

    def __init__(self, root, phonemes = PHONEMES, context=20, partition= "train-clean-100"): # Feel free to add more arguments

        self.context    = context
        self.phonemes   = phonemes

        # TODO: MFCC directory - use partition to acces train/dev directories from kaggle data using root
        self.mfcc_dir       = os.path.join(root,partition,'mfcc')
        # TODO: Transcripts directory - use partition to acces train/dev directories from kaggle data using root
        self.transcript_dir = os.path.join(root,partition,'transcript')

        # TODO: List files in sefl.mfcc_dir using os.listdir in sorted order
        mfcc_names          =  sorted(os.listdir(self.mfcc_dir))
        print(len(mfcc_names))


        # TODO: List files in self.transcript_dir using os.listdir in sorted order
        transcript_names    = sorted(os.listdir(self.transcript_dir))
        # Making sure that we have the same no. of mfcc and transcripts
        assert len(mfcc_names) == len(transcript_names)

        self.mfccs, self.transcripts = [], []

        # TODO: Iterate through mfccs and transcripts
        for i in range(len(mfcc_names)):
        #   Load a single mfcc
            mfcc        = np.load(os.path.join(self.mfcc_dir,mfcc_names[i]))

        #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0))/(np.std(mfcc,axis= 0))

            transcript = np.load(os.path.join(self.transcript_dir,transcript_names[i]))
            transcript = transcript[1:-1]  # Remove [SOS] and [EOS] from the transcript
            # (Is there an efficient way to do this without traversing through the transcript?)
            # Note that SOS will always be in the starting and EOS at end, as the name suggests.
        #   Append each mfcc to self.mfcc, transcript to self.transcript
            self.mfccs.append(mfcc)
            self.transcripts.append(transcript)

        # NOTE:
        # Each mfcc is of shape T1 x 28, T2 x 28, ...
        # Each transcript is of shape (T1+2), (T2+2) before removing [SOS] and [EOS]

        # TODO: Concatenate all mfccs in self.mfccs such that
        # the final shape is T x 28 (Where T = T1 + T2 + ...)
        self.mfccs  = np.concatenate(self.mfccs,axis = 0)


        # TODO: Concatenate all transcripts in self.transcripts such that
        # the final shape is (T,) meaning, each time step has one phoneme output
        self.transcripts    = np.concatenate(self.transcripts,axis=0)
        # Hint: Use numpy to concatenate

        # Length of the dataset is now the length of concatenated mfccs/transcripts
        self.length = len(self.mfccs)

        # Take some time to think about what we have done.
        # self.mfcc is an array of the format (Frames x Features).
        # Our goal is to recognize phonemes of each frame
        # We can introduce context by padding zeros on top and bottom of self.mfcc
        self.mfccs = np.pad(self.mfccs, ((context, context), (0, 0)), 'constant', constant_values=0)# TODO
        print(self.mfccs.shape)
        # The available phonemes in the transcript are of string data type
        # But the neural network cannot predict strings as such.
        # Hence, we map these phonemes to integers

        # TODO: Map the phonemes to their corresponding list indexes in self.phonemes
        self.transcripts = [self.phonemes.index(phon) for phon in self.transcripts]
        # Now, if an element in self.transcript is 0, it means that it is 'SIL' (as per the above example)

    def __len__(self):
        return self.length

    def __getitem__(self, ind):
        start = ind
        end = ind + 2 * self.context + 1
        frames = self.mfccs[start:end]
        # After slicing, you get an array of shape 2*context+1 x 28. But our MLP needs 1d data and not 2d.
        frames = frames.flatten() # TODO: Flatten to get 1d data
        frames      = torch.FloatTensor(frames) # Convert to tensors
        phonemes    = torch.tensor(self.transcripts[ind])

        return frames,phonemes

In [10]:
# Dataset class to load train and validation data

class AudioVALDataset(torch.utils.data.Dataset):

    def __init__(self, root, phonemes = PHONEMES, context=20, partition= "dev-clean"): # Feel free to add more arguments

        self.context    = context
        self.phonemes   = phonemes

        # TODO: MFCC directory - use partition to acces train/dev directories from kaggle data using root
        self.mfcc_dir       = os.path.join(root,partition,'mfcc')
        # TODO: Transcripts directory - use partition to acces train/dev directories from kaggle data using root
        self.transcript_dir = os.path.join(root,partition,'transcript')

        # TODO: List files in sefl.mfcc_dir using os.listdir in sorted order
        mfcc_names  =  sorted(os.listdir(self.mfcc_dir))


        # TODO: List files in self.transcript_dir using os.listdir in sorted order
        transcript_names    = sorted(os.listdir(self.transcript_dir))
        # Making sure that we have the same no. of mfcc and transcripts
        assert len(mfcc_names) == len(transcript_names)

        self.mfccs, self.transcripts = [], []

        # TODO: Iterate through mfccs and transcripts
        for i in range(len(mfcc_names)):
        #   Load a single mfcc
            mfcc = np.load(os.path.join(self.mfcc_dir,mfcc_names[i]))

        #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0))/(np.std(mfcc,axis= 0))
            transcript = np.load(os.path.join(self.transcript_dir,transcript_names[i]))
            transcript = transcript[1:-1]  # Remove [SOS] and [EOS] from the transcript
            # (Is there an efficient way to do this without traversing through the transcript?)
            # Note that SOS will always be in the starting and EOS at end, as the name suggests.
        #   Append each mfcc to self.mfcc, transcript to self.transcript
            self.mfccs.append(mfcc)
            self.transcripts.append(transcript)

        # NOTE:
        # Each mfcc is of shape T1 x 28, T2 x 28, ...
        # Each transcript is of shape (T1+2), (T2+2) before removing [SOS] and [EOS]

        # TODO: Concatenate all mfccs in self.mfccs such that
        # the final shape is T x 28 (Where T = T1 + T2 + ...)
        self.mfccs = np.concatenate(self.mfccs,axis = 0)

        # TODO: Concatenate all transcripts in self.transcripts such that
        # the final shape is (T,) meaning, each time step has one phoneme output
        self.transcripts    = np.concatenate(self.transcripts,axis=0)
        # Hint: Use numpy to concatenate

        # Length of the dataset is now the length of concatenated mfccs/transcripts
        self.length = len(self.mfccs)

        # Take some time to think about what we have done.
        # self.mfcc is an array of the format (Frames x Features).
        # Our goal is to recognize phonemes of each frame
        # We can introduce context by padding zeros on top and bottom of self.mfcc
        self.mfccs = np.pad(self.mfccs, ((context, context), (0, 0)), 'constant', constant_values=0)# TODO

        # The available phonemes in the transcript are of string data type
        # But the neural network cannot predict strings as such.
        # Hence, we map these phonemes to integers

        # TODO: Map the phonemes to their corresponding list indexes in self.phonemes
        self.transcripts = [self.phonemes.index(phon) for phon in self.transcripts]
        # Now, if an element in self.transcript is 0, it means that it is 'SIL' (as per the above example)

    def __len__(self):
        return self.length

    def __getitem__(self, ind):
        end = ind + 2 * self.context+1

        # TODO: Based on context and offset, return a frame at given index with context frames to the left, and right.
        frames = self.mfccs[ind: end]
        # After slicing, you get an array of shape 2*context+1 x 28. But our MLP needs 1d data and not 2d.
        frames = frames.flatten() # TODO: Flatten to get 1d data

        frames      = torch.FloatTensor(frames) # Convert to tensors
        phonemes    = torch.tensor(self.transcripts[ind])

        return frames, phonemes

In [11]:
class AudioTestDataset(torch.utils.data.Dataset):
    def __init__(self, root, context=20, partition= "test-clean"): # Feel free to add more arguments

        self.context = context
        # TODO: MFCC directory - use partition to acces train/dev directories from kaggle data using root
        self.mfcc_dir = os.path.join(root,partition,'mfcc')

        # TODO: List files in sefl.mfcc_dir using os.listdir in sorted order
        mfcc_names =  sorted(os.listdir(self.mfcc_dir))

        self.mfccs= []

        # TODO: Iterate through mfccs and transcripts
        for i in range(len(mfcc_names)):
        #   Load a single mfcc
            mfcc  = np.load(os.path.join(self.mfcc_dir,mfcc_names[i]))
            #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0))/(np.std(mfcc,axis= 0))

            self.mfccs.append(mfcc)


        # Concatenate all mfccs in self.mfccs such that
        self.mfccs  = np.concatenate(self.mfccs,axis = 0)
        # Length of the dataset is now the length of concatenated mfccs/transcripts
        self.length = len(self.mfccs)

        # Take some time to think about what we have done.
        # self.mfcc is an array of the format (Frames x Features).
        # Our goal is to recognize phonemes of each frame
        # We can introduce context by padding zeros on top and bottom of self.mfcc
        self.mfccs = np.pad(self.mfccs, ((context, context), (0, 0)), 'constant', constant_values=0)# TODO



    def __len__(self):
        return self.length

    def __getitem__(self, ind):
        end = ind + 2 * self.context+1

        # TODO: Based on context and offset, return a frame at given index with context frames to the left, and right.
        frames = self.mfccs[ind: end]
        # After slicing, you get an array of shape 2*context+1 x 28. But our MLP needs 1d data and not 2d.
        frames = frames.flatten() # TODO: Flatten to get 1d data

        frames = torch.FloatTensor(frames) # Convert to tensors


        return frames


    # TODO: Create a test dataset class similar to the previous class but you dont have transcripts for this
    # Imp: Read the mfccs in sorted order, do NOT shuffle the data here or in your dataloader.

# Parameters Configuration

Storing your parameters and hyperparameters in a single configuration dictionary makes it easier to keep track of them during each experiment. It can also be used with weights and biases to log your parameters for each experiment and keep track of them across multiple experiments.

In [12]:
config = {
    # Training parameters
    'epochs': 20,
    'batch_size': 8192,
    'context': 25,
    'init_lr':3e-3,

    # Model parameters
    'architecture': 'mlp',
    'dropout': 0.2,
    'time_mask': 10,
    'freq_mask': 5,

    # Scheduler parameters
    'scheduler_type': 'cosine_annealing',  # Options: 'step', 'cosine_annealing', etc.
    'scheduler_params': {
        'T_max': 10,  # Maximum number of epochs (for cosine annealing)
        'eta_min': 1e-5  # Minimum learning rate
    },

    # Optimizer parameters
    'optimizer_type': 'adam',  # Options: 'adam', 'sgd', etc.
    'weight_decay': 1e-4,  # Regularization parameter

    # Weight initialization parameters
    'weight_init': {
        'method': 'kaiming_uniform',  # Options: 'xavier_uniform', 'kaiming_uniform', etc.
        'nonlinearity': 'gelu'  # Activation function for initialization
    }
}


# Create Datasets

In [13]:
root = '/content/11785-f24-hw1p2'
context_size =25
#TODO: Create a dataset object using the AudioDataset class for the training data
train_data = AudioDataset(root = root, phonemes = PHONEMES, context=context_size, partition= "train-clean-100")

# TODO: Create a dataset object using the AudioDataset class for the validation data
val_data = AudioVALDataset(root = root, phonemes = PHONEMES, context=context_size, partition= "dev-clean")

# TODO: Create a dataset object using the AudioTestDataset class for the test data
test_data = AudioTestDataset(root = root, context=context_size, partition= "test-clean")


28539
(36091207, 28)


In [14]:
# Define dataloaders for train, val and test datasets
# Dataloaders will yield a batch of frames and phonemes of given batch_size at every iteration
# We shuffle train dataloader but not val & test dataloader. Why?

train_loader = torch.utils.data.DataLoader(
    dataset     = train_data,
    num_workers = 4,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = True
)

val_loader = torch.utils.data.DataLoader(
    dataset     = val_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False
)

test_loader = torch.utils.data.DataLoader(
    dataset     = test_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False
)


print("Batch size     : ", config['batch_size'])
print("Context        : ", config['context'])
print("Input size     : ", (2*config['context']+1)*28)
print("Output symbols : ", len(PHONEMES))

print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Validation dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Batch size     :  8192
Context        :  25
Input size     :  1428
Output symbols :  42
Train dataset samples = 36091157, batches = 4406
Validation dataset samples = 1928204, batches = 236
Test dataset samples = 1934138, batches = 237


In [15]:
# Testing code to check if your data loaders are working
for i, data in enumerate(train_loader):
    frames, phoneme = data
    print(frames.shape, phoneme.shape)
    break

torch.Size([8192, 1428]) torch.Size([8192])


# Network Architecture


This section defines your network architecture for the homework. We have given you a sample architecture that can easily clear the very low cutoff for the early submission deadline.

In [16]:
import torch.nn as nn  # Import torch.nn and give it the alias 'nn'
import torch.nn.init as init  # Import initialization functions from torch.nn
import torchaudio
import torchaudio.transforms as T

In [17]:
# This architecture will make you cross the very low cutoff
# However, you need to run a lot of experiments to cross the medium or high cutoff
class Network(torch.nn.Module):

    def __init__(self, input_size, output_size):

        super(Network, self).__init__()
        #Version 18

        self.model = torch.nn.Sequential(
            torch.nn.Linear(input_size, 512),
            torch.nn.BatchNorm1d(512),
            torch.nn.GELU(),

            torch.nn.Linear(512, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),

            torch.nn.Linear(1024, 2048),
            torch.nn.BatchNorm1d(2048),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),


            torch.nn.Linear(2048, 3000),
            torch.nn.BatchNorm1d(3000),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),


            torch.nn.Linear(3000, 2048),
            torch.nn.BatchNorm1d(2048),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),


            torch.nn.Linear(2048, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),


            torch.nn.Linear(1024, 512),
            torch.nn.BatchNorm1d(512),
            torch.nn.GELU(),
            torch.nn.Dropout(0.2),



            torch.nn.Linear(512, output_size),
        # #Version 15
        # self.model = torch.nn.Sequential(
        #     torch.nn.Linear(input_size, 512),
        #     torch.nn.BatchNorm1d(512),
        #     torch.nn.LeakyReLU(),


        #     torch.nn.Linear(512, 1024),
        #     torch.nn.BatchNorm1d(1024),
        #     torch.nn.LeakyReLU(),


        #     torch.nn.Linear(1024, 2048),
        #     torch.nn.BatchNorm1d(2048),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),

        #     torch.nn.Linear(2048, 3200),
        #     torch.nn.BatchNorm1d(3200),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),

        #     torch.nn.Linear(3200, 2048),
        #     torch.nn.BatchNorm1d(2048),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),

        #     torch.nn.Linear(2048, 1024),
        #     torch.nn.BatchNorm1d(1024),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),


        #     torch.nn.Linear(1024, 512),
        #     torch.nn.BatchNorm1d(512),
        #     torch.nn.LeakyReLU(),


        #     torch.nn.Linear(512, output_size),
        # Version 10
        # self.model = torch.nn.Sequential(
        #     torch.nn.Linear(input_size, 4096),
        #     torch.nn.BatchNorm1d(4096),
        #     torch.nn.Dropout(0.2),
        #     torch.nn.ReLU(),

        #     torch.nn.Linear(4096, 2048),
        #     torch.nn.BatchNorm1d(2048),
        #     torch.nn.Dropout(0.2),
        #     torch.nn.ReLU(),

        #     torch.nn.Linear(2048, 1024),
        #     torch.nn.BatchNorm1d(1024),
        #     torch.nn.Dropout(0.2),
        #     torch.nn.ReLU(),

        #     torch.nn.Linear(1024, 512),
        #     torch.nn.BatchNorm1d(512),
        #     torch.nn.Dropout(0.2),
        #     torch.nn.ReLU(),

        #     torch.nn.Linear(512, 256),
        #     torch.nn.BatchNorm1d(256),
        #     torch.nn.Dropout(0.2),
        #     torch.nn.ReLU(),

        #     torch.nn.Linear(256, output_size),
        #     # torch.nn.Linear(2048, 1024),
        #     # torch.nn.ReLU(),
        #     # torch.nn.Linear(1024, 512),
        #     # torch.nn.ReLU(),
        #     # torch.nn.Linear(512, 256),
        #     # torch.nn.ReLU(),
        #     # torch.nn.Linear(256, output_size),
        #     # torch.nn.ReLU()
        # )

        # #Version 11
        # self.model = torch.nn.Sequential(
        #     torch.nn.Linear(input_size, 4096),
        #     torch.nn.BatchNorm1d(4096),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),

        #     torch.nn.Linear(4096, 2048),
        #     torch.nn.BatchNorm1d(2048),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),

        #     torch.nn.Linear(2048, 1024),
        #     torch.nn.BatchNorm1d(1024),
        #     torch.nn.Dropout(0.3),
        #     torch.nn.LeakyReLU(),


        #     torch.nn.Linear(1024, output_size),



            # torch.nn.Linear(2048, 1024),
            # torch.nn.ReLU(),
            # torch.nn.Linear(1024, 512),
            # torch.nn.ReLU(),
            # torch.nn.Linear(512, 256),
            # torch.nn.ReLU(),
            # torch.nn.Linear(256, output_size),
            # torch.nn.ReLU()
        )
        # Apply He Initialization to layers
        self._initialize_weights()

    def _initialize_weights(self):
        for layer in self.model:
            if isinstance(layer, nn.Linear):
                # He initialization (Kaiming) for LeakyReLU or ReLU activations
                            # Uniform initialization between -0.1 and 0.1
                nn.init.uniform_(layer.weight, a=-0.1, b=0.1)
                # Optionally initialize bias to zero
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0)



    def forward(self, x):


        # Pass through the model
        out = self.model(x)

        return out





In [18]:
len(frames.to(device))

8192

# Define Model, Loss Function and Optimizer

Here we define the model, loss function, optimizer and optionally a learning rate scheduler.

In [19]:
INPUT_SIZE  = (2*config['context'] + 1) * 28 # Why is this the case?
model       = Network(INPUT_SIZE, len(train_data.phonemes)).to(device)
summary(model, frames.to(device))
# Check number of parameters of your network
# Remember, you are limited to 20 million parameters for HW1 (including ensembles)

----------------------------------------------------------------------------------------------------
Layer                   Kernel Shape         Output Shape         # Params (K)      # Mult-Adds (M)
0_Linear                 [1428, 512]          [8192, 512]               731.65                 0.73
1_BatchNorm1d                  [512]          [8192, 512]                 1.02                 0.00
2_GELU                             -          [8192, 512]                    -                    -
3_Linear                 [512, 1024]         [8192, 1024]               525.31                 0.52
4_BatchNorm1d                 [1024]         [8192, 1024]                 2.05                 0.00
5_GELU                             -         [8192, 1024]                    -                    -
6_Dropout                          -         [8192, 1024]                    -                    -
7_Linear                [1024, 2048]         [8192, 2048]             2,099.20                 2.10

In [20]:
model

Network(
  (model): Sequential(
    (0): Linear(in_features=1428, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=512, out_features=1024, bias=True)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=1024, out_features=2048, bias=True)
    (8): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): GELU(approximate='none')
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=2048, out_features=3000, bias=True)
    (12): BatchNorm1d(3000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): GELU(approximate='none')
    (14): Dropout(p=0.2, inplace=False)
    (15): Linear(in_features=3000, out_features=2048, bias=True)
    (16): BatchNorm1d(2048, 

# Training and Validation Functions

In [21]:
criterion = torch.nn.CrossEntropyLoss() # Defining Loss function.
# We use CE because the task is multi-class classification

optimizer = torch.optim.Adam(model.parameters(), lr= config['init_lr'],weight_decay=config['weight_decay'])  #Defining Optimizer
# Recommended : Define Scheduler for Learning Rate,
# including but not limited to StepLR, MultiStep, CosineAnnealing, CosineAnnealingWithWarmRestarts, ReduceLROnPlateau, etc.
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2,eta_min=1e-4)
# You can refer to Pytorch documentation for more information on how to use them.

# Is your training time very high?
# Look into mixed precision training if your GPU (Tesla T4, V100, etc) can make use of it
# Refer - https://pytorch.org/docs/stable/notes/amp_examples.html

This section covers the training, and validation functions for each epoch of running your experiment with a given model architecture. The code has been provided to you, but we recommend going through the comments to understand the workflow to enable you to write these loops for future HWs.

In [22]:
torch.cuda.empty_cache()
gc.collect()

17

In [23]:
def train(model, dataloader, optimizer, criterion):

    model.train()
    tloss, tacc = 0, 0 # Monitoring loss and accuracy
    batch_bar   = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

    for i, (frames, phonemes) in enumerate(dataloader):

        ### Initialize Gradients
        optimizer.zero_grad()

        ### Move Data to Device (Ideally GPU)
        frames      = frames.to(device)
        phonemes    = phonemes.to(device)

        ### Forward Propagation
        logits  = model(frames)

        ### Loss Calculation
        loss    = criterion(logits, phonemes)

        ### Backward Propagation
        loss.backward()

        ### Gradient Descent
        optimizer.step()

        tloss   += loss.item()
        tacc    += torch.sum(torch.argmax(logits, dim= 1) == phonemes).item()/logits.shape[0]

        batch_bar.set_postfix(loss="{:.04f}".format(float(tloss / (i + 1))),
                              acc="{:.04f}%".format(float(tacc*100 / (i + 1))))
        batch_bar.update()

        #scheduler
        scheduler.step()

        ### Release memory
        del frames, phonemes, logits
        torch.cuda.empty_cache()

    batch_bar.close()
    tloss   /= len(train_loader)
    tacc    /= len(train_loader)

    return tloss, tacc

In [24]:
def eval(model, dataloader):

    model.eval() # set model in evaluation mode
    vloss, vacc = 0, 0 # Monitoring loss and accuracy
    batch_bar   = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')

    for i, (frames, phonemes) in enumerate(dataloader):

        ### Move data to device (ideally GPU)
        frames      = frames.to(device)
        phonemes    = phonemes.to(device)

        # makes sure that there are no gradients computed as we are not training the model now
        with torch.inference_mode():
            ### Forward Propagation
            logits  = model(frames)
            ### Loss Calculation
            loss    = criterion(logits, phonemes)

        vloss   += loss.item()
        vacc    += torch.sum(torch.argmax(logits, dim= 1) == phonemes).item()/logits.shape[0]

        # Do you think we need loss.backward() and optimizer.step() here?

        batch_bar.set_postfix(loss="{:.04f}".format(float(vloss / (i + 1))),
                              acc="{:.04f}%".format(float(vacc*100 / (i + 1))))
        batch_bar.update()

        ### Release memory
        del frames, phonemes, logits
        torch.cuda.empty_cache()

    batch_bar.close()
    vloss   /= len(val_loader)
    vacc    /= len(val_loader)

    return vloss, vacc

# Weights and Biases Setup

This section is to enable logging metrics and files with Weights and Biases. Please refer to wandb documentationa and recitation 0 that covers the use of weights and biases for logging, hyperparameter tuning and monitoring your runs for your homeworks. Using this tool makes it very easy to show results when submitting your code and models for homeworks, and also extremely useful for study groups to organize and run ablations under a single team in wandb.

We have written code for you to make use of it out of the box, so that you start using wandb for all your HWs from the beginning.

In [25]:
wandb.login(key="") #API Key is in your wandb account, under settings (wandb.ai/settings)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [26]:
# Create your wandb run
run = wandb.init(
    name    = "first-run", ### Wandb creates random run names if you skip this field, we recommend you give useful names
    reinit  = True, ### Allows reinitalizing runs when you re-run this cell
    #id     = "y28t31uz", ### Insert specific run id here if you want to resume a previous run
    #resume = "must", ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw1p2", ### Project should be created in your wandb account
    config  = config ### Wandb Config for your run
)

wandb: Currently logged in as: anqiyang00 (anqiyang00-carnegie-mellon-university). Use `wandb login --relogin` to force relogin


In [27]:
### Save your model architecture as a string with str(model)
model_arch  = str(model)

### Save it in a txt file
arch_file   = open("model_arch.txt", "w")
file_write  = arch_file.write(model_arch)
arch_file.close()

### log it in your wandb run with wandb.save()
wandb.save('model_arch.txt')

['/content/wandb/run-20240920_221336-m6hy5ype/files/model_arch.txt']

# Experiment

Now, it is time to finally run your ablations! Have fun!

In [28]:
# Define the checkpoint folder in Google Drive
checkpoint_folder = '/content/drive/MyDrive/checkpoints2/'
os.makedirs(checkpoint_folder, exist_ok=True)  # Create folder if it doesn't exist

# Function to save a checkpoint
def save_checkpoint(epoch, model, optimizer, val_loss, val_acc, best_val_acc, filename):
    checkpoint_path = os.path.join(checkpoint_folder, filename)
    torch.save({
        'epoch': epoch + 1,  # Save the next epoch number
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

# Function to load a checkpoint and resume training
def load_checkpoint(filename, model, optimizer):
    checkpoint_path = os.path.join(checkpoint_folder, filename)

    # Check if the file exists before loading
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        best_val_acc = checkpoint['best_val_acc']
        print(f"Resumed training from checkpoint: {checkpoint_path}, starting at epoch {start_epoch}")
        return start_epoch, best_val_acc
    else:
        print(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")
        return 0, 0.0  # Return default values if no checkpoint is found

# Initialize variables for tracking best accuracy and loss
best_val_acc = 0.0  # To keep track of the best validation accuracy

# Check if there's a checkpoint to resume training from
resume_from_checkpoint = True  # Set this to True if you want to resume from a saved checkpoint
checkpoint_filename = 'best_model.pth'  # The checkpoint filename



if resume_from_checkpoint:
    start_epoch, best_val_acc = load_checkpoint(checkpoint_filename, model, optimizer)
else:
    start_epoch = 0  # Start from scratch if not resuming

<ipython-input-28-db5859482872>:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Resumed training from checkpoint: /content/drive/MyDrive/checkpoints2/best_model.pth, starting at epoch 4


In [29]:
# Function to print model parameters
def print_model_parameters(model):
    for name, param in model.named_parameters():
        if param.requires_grad:  # Only print parameters that require gradients
            print(f"{name}: {param.data}")

# Example usage
print("Current Model Parameters:")
print_model_parameters(model)

Current Model Parameters:
model.0.weight: tensor([[-2.0036e-01,  1.7384e-01,  6.0302e-02,  ..., -9.3578e-02,
         -5.3902e-02,  7.3394e-01],
        [-1.0423e-01, -1.3460e-02,  1.5865e-01,  ...,  1.6057e-01,
          1.2313e-01, -6.0227e-01],
        [-7.5239e-02,  1.4392e-01,  6.9901e-03,  ..., -9.9902e-02,
          2.1128e-03,  1.2550e-01],
        ...,
        [ 2.8548e-02, -8.1949e-03, -3.2411e-02,  ...,  1.1568e-02,
         -1.4952e-02, -2.1755e-01],
        [ 3.3807e-01,  5.5277e-02,  2.7517e-01,  ..., -1.9451e-01,
          4.6580e-03,  5.6659e-01],
        [ 2.2389e+00,  9.2531e-01, -5.0164e-01,  ..., -1.1756e-01,
          1.7541e-02,  6.7572e-01]], device='cuda:0')
model.0.bias: tensor([-6.7523e-03,  2.3919e-02, -9.7993e-03, -1.0775e-02,  1.4852e-02,
         1.0477e-02,  4.4870e-03, -2.3724e-02,  7.7878e-03, -1.3288e-02,
         1.4056e-02,  3.6542e-03,  9.0644e-03, -1.2313e-02,  1.1248e-02,
         1.7376e-02, -1.7885e-02, -4.5955e-04,  3.7972e-03, -1.4723e-02,
   

In [30]:
# Function to print optimizer weight decay
def print_optimizer_weight_decay(optimizer):
    for param_group in optimizer.param_groups:
        print(f"Weight Decay: {param_group['weight_decay']}")

# Example usage
print_optimizer_weight_decay(optimizer)

Weight Decay: 0


In [31]:
# Iterate over number of epochs to train and evaluate your model
torch.cuda.empty_cache()
gc.collect()
wandb.watch(model, log="all")

for epoch in range(config['epochs']):

    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    curr_lr                 = float(optimizer.param_groups[0]['lr'])
    train_loss, train_acc   = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc       = eval(model, val_loader)

    print("\tTrain Acc {:.04f}%\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_acc*100, train_loss, curr_lr))
    print("\tVal Acc {:.04f}%\tVal Loss {:.04f}".format(val_acc*100, val_loss))

    ### Log metrics at each epoch in your run
    # Optionally, you can log at each batch inside train/eval functions
    # (explore wandb documentation/wandb recitation)
    wandb.log({'train_acc': train_acc*100, 'train_loss': train_loss,
               'val_acc': val_acc*100, 'valid_loss': val_loss, 'lr': curr_lr})

    ### Highly Recommended: Save checkpoint in drive and/or wandb if accuracy is better than your current best
    ### Check if validation accuracy has improved
   # Save the best model checkpoint based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(epoch, model, optimizer, val_loss, val_acc, best_val_acc, checkpoint_filename)



Epoch 1/20


Train:   0%|          | 0/4406 [00:00<?, ?it/s]

Val:   0%|          | 0/236 [00:00<?, ?it/s]

	Train Acc 86.8940%	Train Loss 0.3676	 Learning Rate 0.0001241
	Val Acc 84.6956%	Val Loss 0.4687

Epoch 2/20


Train:   0%|          | 0/4406 [00:00<?, ?it/s]

...

Train:   0%|          | 0/4406 [00:00<?, ?it/s]

Val:   0%|          | 0/236 [00:00<?, ?it/s]

	Train Acc 87.4534%	Train Loss 0.3485	 Learning Rate 0.0009873
	Val Acc 84.7153%	Val Loss 0.4750

Epoch 12/20


Train:   0%|          | 0/4406 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Testing and submission to Kaggle

Before we get to the following code, make sure to see the format of submission given in *sample_submission.csv*. Once you have done so, it is time to fill the following function to complete your inference on test data. Refer the eval function from previous cells to get an idea of how to go about completing this function.

In [32]:
def test(model, test_loader):
    ### What you call for model to perform inference?
    model.eval()

    ### List to store predicted phonemes of test data
    test_predictions = []

    ### Which mode do you need to avoid gradients?
    with torch.no_grad(): # TODO

        for i, mfccs in enumerate(tqdm(test_loader)):

            mfccs   = mfccs.to(device)

            logits  = model(mfccs)

            ### Get most likely predicted phoneme with argmax
            predicted_phonemes = torch.argmax(logits,dim=1)
            # Map the predicted indices to the corresponding phoneme names
            predicted_phonemes = [PHONEMES[idx] for idx in predicted_phonemes.cpu().numpy()]

            ### How do you store predicted_phonemes with test_predictions? Hint, look at eval
            test_predictions.extend(predicted_phonemes)

    return test_predictions

In [34]:
predictions = test(model, test_loader)

  0%|          | 0/237 [00:00<?, ?it/s]

Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ad1f0335360>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ad1f0335360>Traceback (most recent call last):

  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1477, in __del__
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1477, in __del__
    self._shutdown_workers()    
self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1460, in _shutdown_workers
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1460, in _shutdown_workers
        if w.is_alive():if w.is_alive():

  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only te

In [35]:
### Create CSV file with predictions
with open("./submission.csv", "w+") as f:
    f.write("id,label\n")
    for i in range(len(predictions)):
        f.write("{},{}\n".format(i, predictions[i]))

In [36]:
### Finish your wandb run
run.finish()

lr,▁▂▂▇▂█▇▄▂▁█
train_acc,▁▁▂▃▅▂▄▆█▅▄
train_loss,██▇▆▄▇▅▃▁▄▅
val_acc,▃▄▁▆▁▃▅▇█▃▄
valid_loss,▂▁▆▁▇▄▄▄▅█▆
lr,0.00099
train_acc,87.45341
train_loss,0.34853
val_acc,84.71531
valid_loss,0.47502


In [37]:
### Submit to kaggle competition using kaggle API (Uncomment below to use)
!kaggle competitions submit -c 11785-hw1p2-f24 -f ./submission.csv -m "Test Submission"

### However, its always safer to download the csv file and then upload to kaggle

100% 19.3M/19.3M [00:00<00:00, 60.2MB/s]
Successfully submitted to 11785 HW1P2 Fall 2024